# Table 1 — Clinical characteristics

Discovery and validation cohort clinical characteristics (side-by-side), plus a patient-level clinical metadata Excel export.

| Item | Detail |
|------|--------|
| **Input** | `adata.uns['case_clinical']` (discovery) and `adata.uns['validation_cohort']` (validation) |
| **IPI/IELSG** | Primary buckets `0–2` / `≥3` (concrete scores + optional component fill from `data/discovery_meta/discovery_clinical_elements.csv`) |
| **Outputs** | `figures/table1/combined_clinical_characteristics_table.*`, validation-only table, and `patient_clinical_metadata.xlsx` (clinical + `case_classifications`) |

Execute cells in order. Set `DLBCL_DATA_DIR` before the setup cell to override the data root.


In [1]:
%matplotlib inline

import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dlbcl.notebook_setup import run_notebook_setup

_ctx = run_notebook_setup('validation_and_discovery', 'table1')
REPO_ROOT = _ctx.repo_root
_paths = _ctx.paths
FIG_DIR = _ctx.fig_dir
adata = _ctx.adata
pred = _ctx.pred
OUTDIR = FIG_DIR
OUTDIR.mkdir(parents=True, exist_ok=True)

import pandas as pd

from dlbcl.dlbcl_io import rel_path, write_supplementary_table
from dlbcl.validation_clinical_table import (
    ClinicalTableConfig,
    LOCATION_ORDER,
    add_location_group,
    audit_validation_clinical_inputs,
    default_discovery_elements_path,
    export_patient_clinical_metadata_xlsx,
    load_validation_metadata,
    render_combined_great_table,
    render_great_table,
    run_combined_clinical_table,
    run_validation_clinical_table,
)

print(f"AnnData: {rel_path(_paths.adata_path, REPO_ROOT)}")
print(f"Figures: {rel_path(FIG_DIR, REPO_ROOT)}")

ADATA_PATH = _paths.adata_path
ELEMENTS_CSV = default_discovery_elements_path(REPO_ROOT)
METADATA_XLSX = OUTDIR / "patient_clinical_metadata.xlsx"


AnnData: DLBCL_location_2026.h5ad
Figures: figures/table1


## Audit validation clinical inputs

In [2]:
meta = load_validation_metadata(source="adata", adata_path=ADATA_PATH)
meta = add_location_group(meta)

audit = audit_validation_clinical_inputs(meta)
print(f"Validation cohort: n={audit.n_patients} ({'OK' if audit.ok else 'ISSUES'})")
print(pd.Series(audit.location_counts).reindex(LOCATION_ORDER).to_string())
if audit.issues:
    print("\nIssues:")
    for issue in audit.issues:
        print(f"  - {issue}")
if audit.warnings:
    print("\nWarnings:")
    for warning in audit.warnings:
        print(f"  - {warning}")
assert audit.ok, audit.issues


Validation cohort: n=303 (OK)
Bone       47
PCNSL     130
Testis     62
Nodal      64

Warnings:
  - IPI/IELSG primary bucket unassignable for 37 patients (excluded from 0–2 / ≥3 rows): {'PCNSL': 15, 'Testis': 13, 'Nodal': 6, 'Bone': 3}


## Combined discovery + validation clinical table

In [3]:
config = ClinicalTableConfig(
    adata_path=ADATA_PATH,
    source="adata",
    output_dir=OUTDIR,
    table_stem="validation_clinical_characteristics_table",
)

discovery_table, validation_table, combined, paths = run_combined_clinical_table(
    config,
    adata=adata,
    repo_root=REPO_ROOT,
    discovery_elements_csv=ELEMENTS_CSV if ELEMENTS_CSV.exists() else None,
)

write_supplementary_table(discovery_table, REPO_ROOT, "discovery_clinical_characteristics_table")
write_supplementary_table(validation_table, REPO_ROOT, "validation_clinical_characteristics_table")
write_supplementary_table(combined, REPO_ROOT, "combined_clinical_characteristics_table")

print("Wrote:")
for key, path in sorted(paths.items()):
    print(f"  {key}: {rel_path(path, REPO_ROOT)}")

render_combined_great_table(combined)


Wrote:
  csv: figures/table1/combined_clinical_characteristics_table.csv
  html: figures/table1/combined_clinical_characteristics_table.html
  pdf: figures/table1/combined_clinical_characteristics_table.pdf
  png: figures/table1/combined_clinical_characteristics_table.png
  svg: figures/table1/combined_clinical_characteristics_table.svg


GT(_tbl_data=                     Characteristic      d_Bone     d_PCNSL    d_Testis  \
0                             Total          20          19           9   
1       Median age (min-max, years)  58 (19–86)  66 (34–78)  68 (53–84)   
2                    Sex female (%)  11 (55.0%)   9 (47.4%)    0 (0.0%)   
3                  Ann Arbor stage:                                       
4                           I(X)B/E           8           0           5   
5                          II(X)A/E           3           0           1   
6                               III           1           0           2   
7                                IV           8          19           1   
8                  IPI/MSKCC-score:                                       
9                               0–2          15           9           7   
10                               ≥3           5          10           2   
11            First-line treatment:                                       
12    HD-MTX-based polychemotherapy           0          16           0   
13                       RCHOP-like          18           0           9   
14       Cell-of-origin (Lymph2CX):                                       
15                              ABC           1          10           2   
16                              GCB          19           6           4   
17                     Intermediate           0           3           3   
18           In situ hybridization:                                       
19                         MYC/BCL2           1           0           0   
20                              MYC  2 (n = 16)  1 (n = 17)   0 (n = 9)   
21                             BCL2  3 (n = 14)   1 (n = 4)   0 (n = 2)   
22                             BCL6  4 (n = 14)   3 (n = 4)   2 (n = 2)   
23                             EBER  0 (n = 20)  0 (n = 19)   0 (n = 9)   

       d_Nodal      v_Bone     v_PCNSL    v_Testis     v_Nodal  
0           16          47         130          62          64  
1   62 (21–81)  61 (18–80)  66 (19–87)  71 (44–89)  66 (30–90)  
2    3 (18.8%)  14 (29.8%)  54 (41.5%)    0 (0.0%)  27 (42.2%)  
3                                                               
4            3          26           0          23          13  
5           11           6           0          11          21  
6            2           0           0           3           9  
7            0          15         130          25          21  
8                                                               
9           14          29          68          28          39  
10           2          15          47          21          19  
11                                                              
12           0           0         130           0           0  
13          16          47           0          62          64  
14                                                              
15           1           2          44          21          14  
16          14          44          59          34          38  
17           1           1          27           7          12  
18                                                              
19           4           1           1           1           0  
20  4 (n = 16)  3 (n = 32)  2 (n = 50)  1 (n = 25)  7 (n = 56)  
21  4 (n = 11)  3 (n = 23)  1 (n = 29)  3 (n = 12)  3 (n = 20)  
22  3 (n = 10)   0 (n = 0)   0 (n = 0)   0 (n = 0)   0 (n = 0)  
23  0 (n = 16)  0 (n = 29)  0 (n = 63)  0 (n = 29)  0 (n = 51)  , _body=<great_tables._gt_data.Body object at 0x1240114f0>, _boxhead=Boxhead([ColInfo(var='Characteristic', type=<ColInfoTypeEnum.default: 1>, column_label='Characteristic', column_align='left', column_width='210px'), ColInfo(var='d_Bone', type=<ColInfoTypeEnum.default: 1>, column_label='Bone', column_align='center', column_width='82px'), ColInfo(var='d_PCNSL', type=<ColInfoTypeEnum.default: 1>, column_label='PCNSL', column_align='center', column_width='82px'), ColInfo(var='d_Testis

## Patient-level clinical metadata Excel (Discovery + Validation tabs)

Clinical columns plus `uns["case_classifications"]` / validation `case_classification_validation` as extra columns (overlapping names kept from clinical).

In [4]:
xlsx_path = export_patient_clinical_metadata_xlsx(
    adata,
    METADATA_XLSX,
    elements_csv=ELEMENTS_CSV if ELEMENTS_CSV.exists() else None,
    repo_root=REPO_ROOT,
)
print(f"Wrote {rel_path(xlsx_path, REPO_ROOT)}")

import openpyxl
wb = openpyxl.load_workbook(xlsx_path, read_only=True)
for sheet in wb.sheetnames:
    ws = wb[sheet]
    n_rows = ws.max_row - 1  # exclude header
    n_cols = ws.max_column
    print(f"  {sheet}: {n_rows} patients × {n_cols} columns")
wb.close()


Wrote figures/table1/patient_clinical_metadata.xlsx
  Discovery: 64 patients × 41 columns
  Validation: 303 patients × 51 columns
